# Graphicalizing Aesop's Fables

This notebook downloads the public-domain Project Gutenberg text for [Æsop's Fables #53103](https://www.gutenberg.org/ebooks/53103), splits it into tale-sized documents, and sends those documents through the ontology-guided semantic pipeline.

The notebook uses the package's default OpenAI `gpt-4.1-mini` client. Set `OPENAI_API_KEY` in the environment before running the model cell. The key is read by the OpenAI SDK and is never stored in this notebook.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from textwrap import wrap

from IPython.display import display

from semantic_graphicalizer import SemanticGraphicalizer, load_aesop_fables

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent

stories = load_aesop_fables(limit=2, cache_dir=ROOT / 'data' / 'raw')
[(story.splitlines()[0], len(story)) for story in stories]

[('THE DAW IN BORROWED FEATHERS', 859), ('THE SUN AND THE WIND', 972)]

In [2]:
len(stories), [story.splitlines()[0] for story in stories]

(2, ['THE DAW IN BORROWED FEATHERS', 'THE SUN AND THE WIND'])

## OpenAI model

The transformer creates the default `OpenAIModelClient` when `model` is omitted. It uses `gpt-4.1-mini` through the Responses API and reads `OPENAI_API_KEY` from the environment.

In [ ]:
from textwrap import wrap

graphicalizer = SemanticGraphicalizer(
    ontology=ROOT / 'configs' / 'ontologies' / 'aesop.yaml',
    prompts=ROOT / 'configs' / 'prompts' / 'aesop.yaml',
)

graphicalizer.fit(stories)
traces = []

for index, story in enumerate(stories, start=1):
    trace = graphicalizer.transform_with_trace([story])[0]
    traces.append(trace)
    title = story.splitlines()[0]
    print(f"\nDocument {index}/{len(stories)}: {title}")
    print("\n".join(wrap(story, width=80)))
    print(f"ID: {trace.document_id}")
    print(f"Entities: {len(trace.entities)} | Relations: {len(trace.relations)} | "
          f"Nodes: {trace.graph.number_of_nodes()} | Edges: {trace.graph.number_of_edges()}")
    print("-" * 80)
    for relation in trace.relations:
        arguments = ", ".join(f"{argument.role}={argument.entity_id}" for argument in relation.arguments)
        print(f"- {relation.type}[{relation.relation}]({arguments})")
    print("\nGraph:")
    display(graphicalizer.display(trace.graph, mode="text"))
    print("=" * 80)


[SemanticGraphicalizer] ready: model=OpenAIModelClient, ontology=aesop-narrative, domain=aesop-narrative
[document-71c1b92da992] segment: 1 -> 1 | 0.1 ms | input_chars=859, chunk_chars=859
[document-71c1b92da992 chunk-0] summarize: 1 -> 1 | 2042.5 ms
[document-71c1b92da992 chunk-0] normalize: 1 -> 1 | 2391.7 ms


In [ ]:
graphicalizer.display(
    traces[0].graph,
    mode="dynamic",
    layout="force",
    show_source=True,
    max_width=34,
    width=1200,
    height=760,
    charge_strength=-320,
    link_distance=150,
    component_spacing=240,
    component_strength=0.18,
)